# Real Madrid ACWR Pipeline — Data Engineering & EDA
**Group A | trAIn Labs | 2024–25 Season**

This notebook ingests raw GPS/IMU tracking data `data/raw/data_acute_vs_chronic.csv` (extracted from `data/data_acute_vs_chronic.zip` if needed) and produces a clean, continuous player-day grid with EWMA-based ACWR signals for all three load metrics.

| Section | Content |
|---|---|
| **0. Setup & Data Loading** | Imports, raw data load, initial inspection |
| **1. Data Cleaning** | Dtype fixes, exercise-type parsing, datetime handling, age computation, player coverage audit, metadata exclusions |
| **2. Outlier Treatment** | Total-distance outlier (player 94884), suspect player profiles (trialists / placeholder metadata) |
| **3. Daily Aggregation** | Period → player-day roll-up, continuous grid, rest-day flagging, session composition diagnostics, player modelability |
| **4. ACWR Computation** | EWMA formulation, per-player computation, squad-wide and per-player diagnostics |
| **5. Persist Outputs** | `full_grid.parquet` (with ACWR), `daily.parquet`, `player_stats.parquet` |

**Final output:** 6,310 rows × 28 columns; 28 players; 66.7% rest days; 18 of 28 players modelable (≥50 active days).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import pickle
import warnings
import subprocess as _sp

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Repo-root resolver (works locally and in Databricks) ──────────────────────
def _repo_root():
    try:
        return Path(_sp.run(
            ["git", "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, check=True,
        ).stdout.strip())
    except Exception:
        p = Path.cwd()
        while p != p.parent:
            if (p / ".git").exists() or (p / "pyproject.toml").exists() or (p / "AGENTS.md").exists():
                return p
            p = p.parent
        return Path.cwd()

REPO_ROOT = _repo_root()
DATA_DIR  = REPO_ROOT / "data"
RAW_CSV   = DATA_DIR / "raw" / "data_acute_vs_chronic.csv"
RAW_ZIP   = DATA_DIR / "data_acute_vs_chronic.zip"

## 0. Setup & Data Loading

Raw dataset: `data/raw/data_acute_vs_chronic.csv` — 3,903 rows × 13 columns, one row per training *period* (drill within a session). Date range: 2024-07-16 to 2025-06-26.

In [2]:
if not RAW_CSV.exists():
    if not RAW_ZIP.exists():
        raise FileNotFoundError(
            f"Missing raw data. Expected {RAW_CSV} or bootstrap archive {RAW_ZIP}."
        )
    import zipfile

    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(RAW_ZIP) as archive:
        archive.extract(RAW_CSV.name, path=RAW_CSV.parent)

df = pd.read_csv(RAW_CSV)

FileNotFoundError: Missing raw data. Expected /Users/rishirajsinharay/Desktop/REAL-MADRID-INTERNSHIP-GROUP-A/data/raw/data_acute_vs_chronic.csv or bootstrap archive /Users/rishirajsinharay/Desktop/REAL-MADRID-INTERNSHIP-GROUP-A/data/data_acute_vs_chronic.zip.

In [ ]:
df.head()

In [ ]:
df.info()

# 1. Data Cleaning & Feature Engineering

The raw dataset contains one row per training *period* (training exercise within a session). We fix data types, clean data, and resolve anomalies before aggregating to daily load.

## 1.1 Fix `is_official_match` and `player_id` types

`is_official_match` is `NaN` for all 2,930 training rows and `1.0` for all 973 match rows — the pattern is unambiguous, so we fill `NaN → 0`. `player_id` is cast to `category` for efficient groupby operations.

In [ ]:
df['is_official_match'] = df['is_official_match'].fillna(0)

# Changing datatype of player_id to object
df['player_id'] = df['player_id'].astype('category')

## 1.2 Parse `exercise_type` from `period_name`

`period_name` encodes drill IDs as `{CATEGORY} {DRILL_ID}` (e.g., `G 1960`, `TAC 0133`). The prefix is the meaningful training category. Match rows always have `NaN` in `period_name`, perfectly correlated with `is_official_match == True`.

In [ ]:
df['period_name'].str.split(' ').str[0].value_counts()

Training categories decoded from the `period_name` prefix:

| Prefix | Category | Count |
|--------|----------|-------|
| `G` | Game-based / small-sided games | 1,184 |
| `TAC` | Tactical | 800 |
| `BP` | *Balón parado* / set pieces | 684 |
| `TEC` | Technical | 262 |
| `NaN` | Official match (no drill logged) | 973 |

All 973 `NaN` entries are match rows — no training periods have a missing `period_name`.

In [ ]:
print("Missing values in period_name when is_official_match is True",df[df['is_official_match']==True].period_name.isna().sum())

print("Missing values in period_name when is_official_match is False",df[df['is_official_match']==False].period_name.isna().sum())

All `NaN` entries in `period_name` correspond exactly to official matches. We label them `"MATCH"` and extract the prefix as a new `exercise_type` column. Individual drill IDs are too granular to model at our dataset scale.

In [ ]:
df['period_name'] = df['period_name'].fillna("MATCH")

df['exercise_type'] = df['period_name'].str.split(' ').str[0]

## 1.3 Parse datetime columns and extract session date

`period_start_time` and `date_of_birth` are stored as ISO 8601 strings. The time component of `period_start_time` is always `00:00:00`, so we extract only the calendar date and drop the original column to avoid redundancy.

In [ ]:
df['period_start_time'] = pd.to_datetime(df['period_start_time']).dt.tz_localize(None).dt.normalize()
df['date_of_birth'] = pd.to_datetime(df['date_of_birth']).dt.tz_localize(None).dt.normalize()

In [ ]:
df.info()

## 1.4 Compute player age at session date

Age is computed as a float (days / 365.25) at each row's specific date, not at a fixed reference date. This ensures the model always receives the player's correct age when predicting load for a future session.

In [ ]:
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
df['age'] = ((df['period_start_time'] - df['date_of_birth']).dt.days / 365.25).round(0)

print(f"Age range: {df['age'].min():.0f} - {df['age'].max():.0f}")
print(f"Age nulls: {df['age'].isna().sum()}")

## 1.5 Player coverage — data sufficiency audit

Before dropping players, we visualise each player's date coverage window. Players appearing only in a narrow preseason or end-of-season block are candidates for exclusion: too few observations to compute a meaningful ACWR and metadata is often incomplete or placeholder.

In [ ]:
def plot_player_date_ranges(df, date_col='period_start_time', player_col='player_id',
                             figsize=(13, 10), show_activity=True):
    
    # Ensure date column is datetime
    dates = pd.to_datetime(df[date_col])
    
    # Get first and last date per player, and sort by first date so the chart reads nicely
    summary = df.assign(_date=dates).groupby(player_col, observed=True).agg(
        first_date=('_date', 'min'),
        last_date=('_date', 'max'),
        n_days=('_date', 'nunique'),
        n_rows=('_date', 'count'),
    ).sort_values('first_date').reset_index()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # One horizontal bar per player showing their activity window
    for i, row in summary.iterrows():
        ax.barh(y=i, 
                width=(row['last_date'] - row['first_date']).days + 1,
                left=row['first_date'],
                height=0.6,
                color='steelblue', alpha=0.3, edgecolor='steelblue')
        
        # Overlay individual activity dates as dots
        if show_activity:
            player_dates = dates[df[player_col] == row[player_col]].unique()
            ax.scatter(player_dates, [i] * len(player_dates),
                       s=8, color='darkblue', alpha=0.7, zorder=3)
    
    # Y-axis: player IDs, ordered by first appearance
    ax.set_yticks(range(len(summary)))
    ax.set_yticklabels([f"{pid} ({n} days)" for pid, n in 
                        zip(summary[player_col], summary['n_days'])])
    ax.set_xlabel('Date')
    ax.set_ylabel('Player ID (active days)')
    ax.set_title(f'Data coverage per player — {len(summary)} players, '
                 f'{dates.min().date()} to {dates.max().date()}')
    
    # Format x-axis with monthly ticks
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    ax.grid(True, axis='x', alpha=0.3)
    ax.invert_yaxis()  # first-appearing player at top
    
    plt.tight_layout()
    plt.show()
    
    return summary


# Run it on your cleaned row-level dataframe (after exclusions)
summary = plot_player_date_ranges(df)
print(summary)

## 1.6 Exclude players with missing metadata

One player (ID 50333) appears only during the first week of preseason with no anthropometric or positional metadata whatsoever. Without height, weight, position, or date of birth, this player cannot be featurised and is excluded via a `dropna` on those columns.

In [ ]:
# Before dropping, document what you're removing and why
affected = df[df['height'].isna()]
print(f"Dropping {len(affected)} rows ({len(affected)/len(df):.2%}) "
      f"from {affected['player_id'].nunique()} player(s) with no metadata")

df = df.dropna(subset=['height', 'weight', 'position_name_en', 'date_of_birth']).reset_index(drop=True)

assert df.isna().sum().sum() == 0 or df.isna().sum()[df.isna().sum() > 0].index.tolist() == []
# or more explicit:
assert df[['player_id', 'height', 'weight', 'position_name_en', 'date_of_birth']].isna().sum().sum() == 0

# 2. Outlier Detection and Treatment

We inspect the numeric column distributions and investigate extreme values before finalising the clean dataset.

In [ ]:
df.describe()

Three anomalies are visible in `describe()`:

- **`total_distance = 32,299 m`** — physiologically impossible for a single period (professional matches are ~10–12 km total).
- **`weight = 200`** — implausible; matches a known placeholder seen in suspect player records.
- **`date_of_birth` min = 1969 / `age` max = 56** — no active first-team player is 56 years old.

We address each anomaly below.

## 2.1 Anomalous `total_distance` — player 94884, 2025-02-15

Only `total_distance` is anomalous for this single MATCH row; the acceleration and HSR values (`acc = 15`, `hsr = 23.5 m`) are within normal match ranges. This points to a GPS export/parsing error on a single field, not a sensor malfunction.

In [ ]:
df.sort_values(by='total_distance', ascending=False).head()

The anomaly is isolated to one row for player 94884 (Forward). We subset this player's official match records to inspect the full distance distribution.

In [ ]:
df_94884 = df[(df['player_id']==94884) & (df['is_official_match']==True)]

In [ ]:
# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Total Distance Distribution and Individual Values for Player 94884 in Official Matches", fontsize=16)

# Histogram for total_distance using Seaborn
sns.histplot(df_94884['total_distance'], bins=25, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Total Distance Histogram")
axes[0].set_xlabel("Total Distance (m)")
axes[0].set_ylabel("Frequency")

# Scatter plot for total_distance using Seaborn
sns.scatterplot(x=df_94884.index, y=df_94884['total_distance'], ax=axes[1], color='salmon', s=50)
axes[1].set_title("Total Distance Scatter Plot")
axes[1].set_xlabel("Data Point Index")
axes[1].set_ylabel("Total Distance (m)")

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

Excluding the outlier, the distribution of match distances for player 94884 is unimodal and well-behaved. The **median** is used as the replacement value — it is robust to skew and avoids inventing a point estimate that coaches would treat as real data.

In [ ]:
df_94884_filtered = df_94884[df_94884['total_distance'] <= 20000]

# Calculate mean, median, mode for the filtered data
mean_distance = df_94884_filtered['total_distance'].mean()
median_distance = df_94884_filtered['total_distance'].median()
mode_distance = df_94884_filtered['total_distance'].mode()[0] # .mode() can return multiple values, so take the first

print(f"Mean of total_distance (excluding outlier): {mean_distance:.2f} m")
print(f"Median of total_distance (excluding outlier): {median_distance:.2f} m")
print(f"Mode of total_distance (excluding outlier): {mode_distance:.2f} m")

# Plot histogram for the filtered data using Seaborn
plt.figure(figsize=(10, 6))
sns.histplot(df_94884_filtered['total_distance'], bins=25, kde=True, color='purple')
plt.title("Total Distance Distribution for Player 94884 (Outlier Excluded)")
plt.xlabel("Total Distance (m)")
plt.ylabel("Frequency")
plt.show()

Replace the anomalous value with the player's median match distance (computed above) and verify the fix.

In [ ]:
df.loc[(df['player_id'] == 94884) & (df['total_distance'] >= 20000), 'total_distance'] = median_distance

# Verify the change
print("Value after replacement:")
print(df[(df['player_id'] == 94884) & (df['period_start_time'] == '2025-02-15')][['total_distance']])

In [ ]:
# Verify the bad row is actually fixed
print(df.loc[(df['player_id'] == 94884) & (df['period_start_time'] == '2025-02-15'), 
             ['total_distance', 'acc_band7plus_total_effort_count', 
              'velocity_band6plus7_total_distance']])

# Check: any remaining total_distance > 20000 anywhere?
print(f"Rows with total_distance > 20000: {(df['total_distance'] > 20000).sum()}")

## 2.2 Suspect Player Profiles — Trialists and Placeholder Metadata

The 1969 birth year and `weight = 200` placeholder flag several players as likely trialists or academy call-ups whose metadata was not properly populated. We audit each before deciding to exclude or retain.

In [ ]:
weight_200 = df[df['weight']==200].player_id.unique()

weight_200

In [ ]:
for pid in weight_200:
    rows = df[df['player_id'] == pid]
    print(f"Player {pid}: {len(rows)} rows, "
          f"{rows['period_start_time'].min().date()} to {rows['period_start_time'].max().date()}, "
          f"DOB: {rows['date_of_birth'].iloc[0].date()}")

In [ ]:
# Drop preseason/end-of-season trialists with placeholder biometrics or insufficient data
trialists_to_drop = weight_200
before = len(df)
df = df[~df['player_id'].isin(trialists_to_drop)].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows across {len(trialists_to_drop)} players")

# Sanity check — no more placeholder biometrics anywhere
assert ((df['height'] == 180) & (df['weight'] == 200)).sum() == 0
assert df['weight'].max() <= 100, "Unexpectedly high weight remaining"
assert df['date_of_birth'].dt.year.min() >= 1980, "Unexpectedly old DOB remaining"
print(f"Final dataset: {len(df)} rows, {df['player_id'].nunique()} players")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

targets = ["total_distance", "velocity_band6plus7_total_distance", "acc_band7plus_total_effort_count"]
labels  = ["Total Distance (m)", "Vel Total (m)", "Acc Total (count)"]

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

for i, (col, label) in enumerate(zip(targets, labels)):
    vals     = df[col].dropna()
    log_vals = np.log1p(vals)

    # raw histogram
    ax1 = fig.add_subplot(gs[0, i])
    ax1.hist(vals, bins=60, color="#C0392B", edgecolor="white", linewidth=0.3)
    ax1.set_title(f"{label}\n(raw)", fontsize=10)
    ax1.set_xlabel(label, fontsize=8)
    ax1.set_ylabel("Frequency", fontsize=8)
    ax1.axvline(vals.mean(),   color="black",  lw=1.5, linestyle="--", label=f"mean={vals.mean():.1f}")
    ax1.axvline(vals.median(), color="gold",   lw=1.5, linestyle=":",  label=f"med={vals.median():.1f}")
    ax1.legend(fontsize=7)

    # log1p histogram
    ax2 = fig.add_subplot(gs[1, i])
    ax2.hist(log_vals, bins=60, color="#2980B9", edgecolor="white", linewidth=0.3)
    ax2.set_title(f"{label}\n(log1p)", fontsize=10)
    ax2.set_xlabel("log1p value", fontsize=8)
    ax2.set_ylabel("Frequency", fontsize=8)
    ax2.axvline(log_vals.mean(),   color="black", lw=1.5, linestyle="--", label=f"mean={log_vals.mean():.2f}")
    ax2.axvline(log_vals.median(), color="gold",  lw=1.5, linestyle=":",  label=f"med={log_vals.median():.2f}")
    ax2.legend(fontsize=7)

plt.suptitle("Target Variable Distributions — Active Days Only (n=2,103)", fontsize=13, fontweight="bold")
plt.show()
print("Skewness:", {col: round(df[col].skew(), 2) for col in targets})


# 3. Daily Aggregation

The raw data is at *period level* (one row per drill within a session). ACWR requires *daily* load totals. We aggregate to one row per `(player_id, date)`, preserving session composition via pivoted period counts and per-type load columns.

In [ ]:
EXERCISE_TYPES = df.exercise_type.unique()
LOAD_COLS = ['total_distance', 'acc_band7plus_total_effort_count', 'velocity_band6plus7_total_distance']

## 3.1 Aggregate to daily

For each player-day:
- **Load totals** (`total_distance`, `acc_total`, `hsr_total`) — summed across all periods.
- **Exercise type set** (`exercise_types`) — frozenset of exercise categories that occurred (e.g. `{'G', 'TAC', 'BP'}`); preserves session composition without creating per-type columns.
- **Static metadata** (position, height, weight, DOB) — `first` value (constant per player).

In [ ]:
def aggregate_to_daily(df):
    # Roll up drill-level periods to one row per (player, date); exercise types as frozenset
    # 1. Daily totals across all periods
    totals = df.groupby(['player_id', 'period_start_time'], observed=True).agg(
        total_distance=('total_distance', 'sum'),
        acc_total=('acc_band7plus_total_effort_count', 'sum'),
        vel_total=('velocity_band6plus7_total_distance', 'sum'),
        is_match=('is_official_match', 'max'),
        n_periods=('activity_id', 'count'),
    ).reset_index()

    # 2. Set of exercise types that occurred on each player-day
    exercise_sets = (
        df.groupby(['player_id', 'period_start_time'], observed=True)['exercise_type']
        .apply(frozenset)
        .reset_index()
        .rename(columns={'exercise_type': 'exercise_types'})
    )

    # 3. Static per-player metadata (constant within a player)
    static = df.groupby('player_id', observed=True).agg(
        position_name_en=('position_name_en', 'first'),
        height=('height', 'first'),
        weight=('weight', 'first'),
        date_of_birth=('date_of_birth', 'first'),
    ).reset_index()

    # 4. Merge everything on player-day
    daily = (
        totals
        .merge(exercise_sets, on=['player_id', 'period_start_time'], how='left')
        .merge(static,        on='player_id',           how='left')
    )

    # 5. Recompute age at each row's date
    daily['age'] = (daily['period_start_time'] - daily['date_of_birth']).dt.days / 365.25

    # 6. Sanity check
    assert len(daily) == df.groupby(['player_id', 'period_start_time'], observed=True).ngroups, "Row count mismatch"

    return daily.sort_values(['player_id', 'period_start_time']).reset_index(drop=True)


daily = aggregate_to_daily(df)
print(f"Shape: {daily.shape}")
print(f"Columns: {daily.columns.tolist()}")
daily.head()

## Section 4 — Modeling Input

The model answers one question: **given a planned session for a given player on a given day, what load will it produce?**

This is a cross-sectional regression, not a time-series forecast. Features describe the session about to happen, not the load history leading up to it.

### Feature set (~20 features)

| Block | Features | Rationale |
|---|---|---|
| Player identity | `age`, `height`, `weight` + 5 position one-hots | Same session produces different load for different players |
| Session design | `has_G`, `has_TAC`, `has_BP`, `has_TEC`, `has_MATCH`, `n_session_types` | The coach input — what's the session |
| Calendar context | 7 day-of-week one-hots, `weeks_since_start` | Pre-season is heavier; weekday/weekend differs |

### Targets (3 separate models)

- `total_distance`
- `acc_total`
- `vel_total`

In [ ]:
import pandas as pd

# Block: Session design — derived from the exercise_types frozenset
def add_session_features(df, exercise_types_col='exercise_types'):
    out = df.copy()
    types_series = df[exercise_types_col]
    for etype in ['G', 'TAC', 'BP', 'TEC', 'MATCH']:
        out[f'has_{etype}'] = types_series.apply(lambda s: int(etype in s))
    out['n_session_types'] = types_series.apply(len)
    return out


# Block: Player attributes — one-hot encode position
def add_position_features(df):
    out = df.copy()
    POSITIONS = ['Central Back', 'Central Midfielder', 'Forward', 'Full Back', 'Winger']
    for pos in POSITIONS:
        col_name = 'pos_' + pos.lower().replace(' ', '_')
        out[col_name] = (out['position_name_en'] == pos).astype(int)
    return out


SEASON_START_DATE = pd.Timestamp('2024-07-15')

def add_calendar_features(df, season_start=SEASON_START_DATE):
    out = df.copy()
    out['days_since_start'] = (out['period_start_time'] - season_start).dt.days.astype(int)
    return out


def add_activity_history_features(df, max_lookback=21):
    """
    Add days_since_last_activity and days_since_last_match per player.
    Both are computed within each player's timeline, capped at max_lookback,
    and use only PAST activity (not today's).
    """
    out = df.sort_values(['player_id', 'period_start_time']).reset_index(drop=True).copy()
    
    results_dsla, results_dslm = [], []
    for pid, g in out.groupby('player_id', observed=True, sort=False):
        # Days since last ACTIVE day (any session) — exclude today, look only at past
        # Since `daily` only contains active rows, every row IS an active day.
        # We need the gap between consecutive rows for the same player.
        prev_active_date = g['period_start_time'].shift(1)
        days_since_active = (g['period_start_time'] - prev_active_date).dt.days
        results_dsla.append(days_since_active)
        
        # Days since last MATCH — look at past matches only
        match_dates = g['period_start_time'].where(g['has_MATCH'] == 1)
        last_match = match_dates.shift(1).ffill()  # shift first to exclude today's match
        days_since_match = (g['period_start_time'] - last_match).dt.days
        results_dslm.append(days_since_match)
    
    out['days_since_last_activity'] = pd.concat(results_dsla).sort_index()
    out['days_since_last_match']    = pd.concat(results_dslm).sort_index()
    
    # First-row-per-player will have NaN. Fill with the cap.
    out['days_since_last_activity'] = out['days_since_last_activity'].fillna(max_lookback).clip(upper=max_lookback)
    out['days_since_last_match']    = out['days_since_last_match'].fillna(max_lookback).clip(upper=max_lookback)
    
    return out

In [ ]:
model_data = daily.copy()
model_data = add_session_features(model_data)
model_data = add_position_features(model_data)
model_data = add_calendar_features(model_data)
model_data = add_activity_history_features(model_data)

print(f"Shape: {model_data.shape}")

# Quick sanity check on the new features
print("\nNew feature distributions:")
for col in ['days_since_start', 'days_since_last_activity', 'days_since_last_match']:
    print(f"  {col}: range [{model_data[col].min()}, {model_data[col].max()}], median {model_data[col].median()}")

In [ ]:
model_data

In [ ]:
# Drop bookkeeping columns we don't need anymore
DROP_COLS = [
    #'player_id',
    'period_start_time',  # already encoded in days since start
    'is_match',           # replaced by has_MATCH
    'n_periods',          # artifact of period-level aggregation
    'exercise_types',     # replaced by has_* flags (and frozenset doesn't serialize)
    'position_name_en',   # replaced by pos_* one-hots
    'date_of_birth',      # already encoded in age
    'archetype_str',      # display-only, not a feature
]
to_drop = [c for c in DROP_COLS if c in model_data.columns]
model_data = model_data.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} bookkeeping columns")
print(f"Final shape: {model_data.shape}")
print(f"\nColumns:")
for c in model_data.columns:
    print(f"  {c}")

In [ ]:
# Final counts: 2 IDs + 3 targets + 22 features = 27 columns
TARGETS = ['total_distance', 'acc_total', 'vel_total']
NON_FEATURE_COLS = ['period_start_time'] + TARGETS

feature_cols = [c for c in model_data.columns if c not in NON_FEATURE_COLS]
print(f"\n{len(feature_cols)} features:")
for c in feature_cols:
    print(f"  {c}")

# NaN check — should be zero
nan_counts = model_data.isna().sum()
print(f"\nNaN counts: {nan_counts[nan_counts > 0].to_dict() if (nan_counts > 0).any() else 'none'}")

In [ ]:
OUT = DATA_DIR / "processed"
OUT.mkdir(parents=True, exist_ok=True)
model_data.to_parquet(OUT / 'model_data.parquet', index=False)

# Verify roundtrip
md_check = pd.read_parquet(OUT / 'model_data.parquet')
assert md_check.shape == model_data.shape
print(f"Saved: {model_data.shape} → {(OUT/'model_data.parquet').stat().st_size / 1024:.1f} KB")